# MNN-Pocket-0.3B: LoRA-пилот под чат-менеджера

**Что делает ноутбук**
1. меряет базовую модель на 30 эталонных вопросах (E1);
2. учит LoRA на 2174 сгенерированных парах (~10–15 мин на T4);
3. меряет обученную модель на тех же вопросах (E2);
4. складывает адаптер в zip и предлагает скачать.

**Как запускать:** `Runtime → Change runtime type → T4 GPU`, затем `Runtime → Run all`.
В конце будет строка `RESULT_JSON={...}` — пришлите её, по ней соберу отчёт.


In [ ]:
!nvidia-smi


Sun Nov  8 19:22:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip -q install -U "transformers>=4.51" peft datasets accelerate 2>&1 | tail -3


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [ ]:
!git clone -q -b chats-v2 https://github.com/levushka994-lab/moneysuper /content/repo
%cd /content/repo/tools/local-model
!wc -l dataset.jsonl val.jsonl golden.jsonl


/content/repo/tools/local-model
   2174 dataset.jsonl
    511 val.jsonl
     30 golden.jsonl
   2715 total


In [ ]:
import json, re, time, statistics
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "taobao-mnn/MNN-Pocket-0.3B"
tok = AutoTokenizer.from_pretrained(MODEL)
tok.chat_template = open("chat_template.jinja", encoding="utf-8").read()
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="cuda")
model.eval()
print("params:", sum(p.numel() for p in model.parameters()) / 1e6, "M")


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/1.36k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.41k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.10M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/654k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.77M [00:00<?, ?B/s]

... (5 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

params: 302.830976 M


In [ ]:
from prompts import build_system_prompt

SYS = build_system_prompt()
CASES = [json.loads(l) for l in open("golden.jsonl", encoding="utf-8")]

CAT_NUMS = set(range(101)) | {390, 35, 12, 150, 48}
for it in json.load(open("catalog.json", encoding="utf-8"))["items"]:
    CAT_NUMS |= {it["price"], it.get("min", 0), it.get("stock", 0)}


def nums(s):
    out = set()
    for m in re.findall(r"\d[\d\s\u202f]*", s):
        try:
            out.add(int(m.replace(" ", "").replace("\u202f", "")))
        except ValueError:
            pass
    return out


@torch.no_grad()
def gen(q, m, sysp=None, max_new=100):
    sysp = sysp or SYS
    msgs = [{"role": "system", "content": sysp}, {"role": "user", "content": q}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = tok(text, return_tensors="pt").to(m.device)
    t0 = time.time()
    out = m.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.eos_token_id)
    dt = time.time() - t0
    ans = tok.decode(out[0][enc["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    return ans, dt


def evaluate(m, cases, tag, sysp=None):
    rows = []
    for c in cases:
        a, dt = gen(c["q"], m, sysp)
        got, req = nums(a), set(c["facts"])
        rows.append({
            "id": c["id"], "ans": a, "dt": dt,
            "fact": req <= got,
            "halluc": bool(got - req - nums(c["q"]) - CAT_NUMS),
            "esc": bool(re.search(r"менеджер|уточн|передам", a, re.I)) == bool(c.get("escalate")),
            "words": len(a.split()),
            "style": bool(re.search(r"(?i)(отличн\w+ вопрос|конечно[,!.]|рад[а]? помочь|не просто\b[^.!?]{1,40}\bа\b|давай(те)? (разбер|погруз)|важно отметить|надеюсь, (это|данная))", a)),
            "zh": bool(re.search(r"[\u4e00-\u9fff]", a)),
        })
    n = len(rows)
    res = {
        "tag": tag, "n": n,
        "fact": round(sum(r["fact"] for r in rows) / n, 3),
        "halluc": round(sum(r["halluc"] for r in rows) / n, 3),
        "brevity": round(sum(r["words"] <= 45 for r in rows) / n, 3),
        "escalation": round(sum(r["esc"] for r in rows) / n, 3),
        "style": round(sum(r["style"] for r in rows) / n, 3),
        "chinese": sum(r["zh"] for r in rows),
        "p50_s": round(statistics.median(r["dt"] for r in rows), 2),
    }
    print(json.dumps(res, ensure_ascii=False))
    return res, rows


BASE, BASE_ROWS = evaluate(model, CASES, "base")
for r in BASE_ROWS[:5]:
    print(r["id"], "|", r["ans"][:160])


## Обучение LoRA

Датасет уже содержит system/user/assistant; промпт маскируется (`-100`), учится только ответ.


In [ ]:
from datasets import load_dataset

raw = load_dataset("json", data_files={"train": "dataset.jsonl", "validation": "val.jsonl"})


def prep(ex):
    msgs = ex["messages"]
    p = tok.apply_chat_template(msgs[:2], tokenize=False, add_generation_prompt=True)
    full = p + msgs[2]["content"] + tok.eos_token
    pids = tok(p, add_special_tokens=False)["input_ids"]
    ids = tok(full, add_special_tokens=False, truncation=True, max_length=640)["input_ids"]
    lab = list(ids)
    k = min(len(pids), len(lab))
    lab[:k] = [-100] * k
    return {"input_ids": ids, "labels": lab}


train_ds = raw["train"].map(prep, remove_columns=raw["train"].column_names)
val_ds = raw["validation"].map(prep, remove_columns=raw["validation"].column_names)
print(len(train_ds), len(val_ds), "max len", max(len(x) for x in train_ds["input_ids"]))


In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()


class Coll:
    def __init__(self, pad):
        self.pad = pad

    def __call__(self, feats):
        L = max(len(f["input_ids"]) for f in feats)
        return {
            "input_ids": torch.tensor([f["input_ids"] + [self.pad] * (L - len(f["input_ids"])) for f in feats]),
            "labels": torch.tensor([f["labels"] + [-100] * (L - len(f["labels"])) for f in feats]),
            "attention_mask": torch.tensor([[1] * len(f["input_ids"]) + [0] * (L - len(f["input_ids"])) for f in feats]),
        }


args = TrainingArguments(
    output_dir="/content/out",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_strategy="no",
    report_to="none",
    max_grad_norm=1.0,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=Coll(tok.pad_token_id))
t0 = time.time()
trainer.train()
print("train minutes:", round((time.time() - t0) / 60, 1))


In [ ]:
model.eval()
TUNED, TUNED_ROWS = evaluate(model, CASES, "tuned")
for r in TUNED_ROWS[:5]:
    print(r["id"], "|", r["ans"][:160])

summary = {"base": BASE, "tuned": TUNED, "train_min": round((time.time() - t0) / 60, 1)}
print("RESULT_JSON=" + json.dumps(summary, ensure_ascii=False))


In [ ]:
model.save_pretrained("/content/adapter")
tok.save_pretrained("/content/adapter")
!cd /content && zip -qr adapter.zip adapter && du -h adapter.zip
print('adapter ready at /content/adapter.zip')
